In [ ]:
import shutil, os

src_dir = "/content/Big_Data_Week_12_Notebook"
for fname in ["week12_transactions.csv", "week12_api_sample.json"]:
    src = os.path.join(src_dir, fname)
    if os.path.exists(src):
        shutil.copy(src, fname)
        print(f"Copied {fname} ✅")
    else:
        print(f"NOT FOUND: {src}")

print("\nNow in /content:", [f for f in os.listdir('.') if f.endswith(('.csv','.json'))])

Copied week12_transactions.csv ✅
Copied week12_api_sample.json ✅

Now in /content: ['week12_api_sample.json', 'week12_transactions.csv']


In [ ]:
import sqlite3, pandas as pd
df = pd.read_csv("week12_transactions.csv")
conn = sqlite3.connect("momo.db")
df.to_sql("transactions", conn, if_exists="replace", index=False)
print(pd.read_sql("SELECT COUNT(*) AS n FROM transactions", conn))

      n
0  2000


In [ ]:
import sqlite3, pandas as pd
conn = sqlite3.connect('momo.db')

print('=== Per txn_type ===')
q1 = '''
SELECT txn_type,
       COUNT(*)                  AS n_txns,
       SUM(amount_rwf)           AS total_rwf,
       ROUND(AVG(amount_rwf), 2) AS avg_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_rwf DESC
'''
print(pd.read_sql(q1, conn).to_string(index=False))

print()
print('=== Per district ===')
q2 = '''
SELECT district, SUM(amount_rwf) AS total_rwf
FROM transactions
GROUP BY district
ORDER BY total_rwf DESC
'''
print(pd.read_sql(q2, conn).to_string(index=False))

=== Per txn_type ===
txn_type  n_txns  total_rwf  avg_rwf
    send     868   62049000 71485.02
 paybill     330   29970000 90818.18
 receive     489   27024000 55263.80
 airtime     313     833000  2661.34

=== Per district ===
  district  total_rwf
   Muhanga   21076000
    Gasabo   20738000
      Huye   18736000
Nyarugenge   18091000
  Kicukiro   11338000
   Musanze   10490500
    Rubavu   10110500
 Nyagatare    9296000


In [ ]:
q_hour = """
SELECT hour, COUNT(*) AS n
FROM transactions
GROUP BY hour
ORDER BY hour
"""
sql_hour = pd.read_sql(q_hour, conn)
print(sql_hour.to_string(index=False))

 hour   n
    0  33
    1  32
    2  34
    3  37
    4  34
    5  33
    6  33
    7  25
    8 168
    9 144
   10 168
   11 169
   12 170
   13  35
   14  34
   15  35
   16  38
   17 183
   18 176
   19 158
   20 185
   21  26
   22  25
   23  25


In [ ]:
pandas_hour = (df.groupby("hour")
                 .size()
                 .reset_index(name="n")
                 .sort_values("hour"))
print(pandas_hour.to_string(index=False))

 hour   n
    0  33
    1  32
    2  34
    3  37
    4  34
    5  33
    6  33
    7  25
    8 168
    9 144
   10 168
   11 169
   12 170
   13  35
   14  34
   15  35
   16  38
   17 183
   18 176
   19 158
   20 185
   21  26
   22  25
   23  25


In [ ]:
# Merge on hour and compare
merged = sql_hour.merge(pandas_hour, on="hour", suffixes=("_sql", "_pandas"))
merged["match"] = merged["n_sql"] == merged["n_pandas"]
print("\nAll hours match:", merged["match"].all())
print("Busiest hour:", merged.loc[merged["n_sql"].idxmax(), "hour"])
print("Max txns:", merged["n_sql"].max())


All hours match: True
Busiest hour: 20
Max txns: 185


In [ ]:
import json

with open("week12_api_sample.json") as f:
    sample = json.load(f)

temps = sample["hourly"]["temperature_2m"]
times = sample["hourly"]["time"]

print("Sample file hours:", len(times))
print("Sample max:", max(temps))          # expect 27.9
print("Sample min:", min(temps))          # expect 15.9
print("Hottest idx:", temps.index(max(temps)))
print("Hottest time:", times[temps.index(max(temps))])  # expect 2026-11-22T14:00
print("Hours ≥ 25:", sum(t >= 25 for t in temps))       # expect 7

Sample file hours: 24
Sample max: 27.9
Sample min: 15.9
Hottest idx: 14
Hottest time: 2026-11-22T14:00
Hours ≥ 25: 7


In [ ]:
import json, requests

try:
    url = ("https://api.open-meteo.com/v1/forecast"
           "?latitude=-1.9441&longitude=30.0619"
           "&hourly=temperature_2m&forecast_days=1&timezone=Africa%2FKigali")
    data = requests.get(url, timeout=10).json()
    print("Source: LIVE API ✅")
except Exception as e:
    print(f"Live API failed ({e}) → using offline sample")
    with open("week12_api_sample.json") as f:
        data = json.load(f)

# Extract
times = data["hourly"]["time"]
temps = data["hourly"]["temperature_2m"]

# Report
print("Hours:", len(times))
print("Max temp:", max(temps), "°C")
print("Min temp:", min(temps), "°C")

# Hottest hour — using .index(max(...))
hottest_idx = temps.index(max(temps))
print("Hottest at:", times[hottest_idx])

Source: LIVE API ✅
Hours: 24
Max temp: 24.1 °C
Min temp: 16.0 °C
Hottest at: 2026-11-24T14:00


In [ ]:
import pandas as pd

df_w = pd.DataFrame({"time": times, "temp_c": temps})

print(df_w.describe())

# Mask: hours ≥ 25 °C
hot = df_w[df_w["temp_c"] >= 25]
print("\nHours ≥ 25 °C:", len(hot))

          temp_c
count  24.000000
mean   18.891667
std     2.515157
min    16.000000
25%    16.675000
50%    18.500000
75%    20.300000
max    24.100000

Hours ≥ 25 °C: 0


In [ ]:
import json, pandas as pd

# Use offline sample so results are reproducible
with open("week12_api_sample.json") as f:
    data = json.load(f)

times = data["hourly"]["time"]
temps = data["hourly"]["temperature_2m"]

print("Hours:", len(times))
print("Max:", max(temps), "°C")
print("Min:", min(temps), "°C")
idx = temps.index(max(temps))
print("Hottest at:", times[idx])
print("Hours ≥ 25 °C:", sum(1 for t in temps if t >= 25))

df_w = pd.DataFrame({"time": times, "temp_c": temps})
print(df_w.describe())

Hours: 24
Max: 27.9 °C
Min: 15.9 °C
Hottest at: 2026-11-22T14:00
Hours ≥ 25 °C: 7
          temp_c
count  24.000000
mean   21.137500
std     4.104696
min    15.900000
25%    17.650000
50%    19.900000
75%    25.025000
max    27.900000


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("week12")
         .master("local[*]")
         .getOrCreate())

sdf = spark.read.csv("week12_transactions.csv", header=True, inferSchema=True)

print("Rows:", sdf.count())     # expect 2000
sdf.printSchema()
sdf.show(5)

Rows: 2000
root
 |-- txn_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)

+------+----------+----+----------+--------+----------+-------+
|txn_id|      date|hour|  district|txn_type|amount_rwf|flagged|
+------+----------+----+----------+--------+----------+-------+
|T00001|2026-11-01|   8|    Gasabo|    send|      1000|      0|
|T00002|2026-11-09|  12|Nyarugenge|    send|     25000|      0|
|T00003|2026-11-19|   8|   Muhanga|    send|     51000|      0|
|T00004|2026-11-01|  17|      Huye|    send|     79000|      0|
|T00005|2026-11-15|  12|      Huye|    send|    109000|      0|
+------+----------+----+----------+--------+----------+-------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import col

# How many txns above 100,000 RWF?
n_above = sdf.filter(col("amount_rwf") > 100000).count()
print("Transactions > 100,000 RWF:", n_above)     # expect 451

# Show 5 flagged transactions
sdf.filter(col("flagged") == 1).show(5)

Transactions > 100,000 RWF: 451
+------+----------+----+--------+--------+----------+-------+
|txn_id|      date|hour|district|txn_type|amount_rwf|flagged|
+------+----------+----+--------+--------+----------+-------+
|T00279|2026-11-29|  19| Muhanga|    send|    127000|      1|
|T00326|2026-11-07|  22|    Huye|    send|    127000|      1|
|T00536|2026-11-27|  17|  Gasabo|    send|    127000|      1|
|T00655|2026-11-30|  17|  Rubavu|    send|    134000|      1|
|T00697|2026-11-01|   9| Muhanga|    send|    127000|      1|
+------+----------+----+--------+--------+----------+-------+
only showing top 5 rows


sdf.groupBy("txn_type") \
   .sum("amount_rwf") \
   .withColumnRenamed("sum(amount_rwf)", "total_rwf") \
   .orderBy("total_rwf", ascending=False) \
   .show()

In [ ]:
dist_pd = (sdf.groupBy("district")
              .sum("amount_rwf")
              .withColumnRenamed("sum(amount_rwf)", "total_rwf")
              .orderBy("total_rwf", ascending=False)
              .toPandas())
print(dist_pd.to_string(index=False))

  district  total_rwf
   Muhanga   21076000
    Gasabo   20738000
      Huye   18736000
Nyarugenge   18091000
  Kicukiro   11338000
   Musanze   10490500
    Rubavu   10110500
 Nyagatare    9296000


In [ ]:
sdf.groupBy("txn_type") \
   .sum("amount_rwf") \
   .withColumnRenamed("sum(amount_rwf)", "total_rwf") \
   .orderBy("total_rwf", ascending=False) \
   .show()

+--------+---------+
|txn_type|total_rwf|
+--------+---------+
|    send| 62049000|
| paybill| 29970000|
| receive| 27024000|
| airtime|   833000|
+--------+---------+



In [ ]:
# Part 4

In [ ]:
  #Scenario A: You have a 2,000-row CSV and need a quick average.
   #Scenario B: You need to aggregate 10 TB of logs across 50 servers.
    #Scenario C: A table with 5 million rows sits in PostgreSQL.
    #Scenario D: You need today's exchange rate from a web service.

In [1]:
!pip install pyspark -q

In [3]:
import shutil, os

src = "/content/Big_Data_Week_12_Notebook/week12_transactions.csv"
shutil.copy(src, "week12_transactions.csv")

# Also copy the JSON
shutil.copy("/content/Big_Data_Week_12_Notebook/week12_api_sample.json",
            "week12_api_sample.json")

print("Files in cwd:", [f for f in os.listdir('.') if f.endswith(('.csv','.json'))])

Files in cwd: ['week12_api_sample.json', 'week12_transactions.csv']


In [4]:
import pandas as pd
df = pd.read_csv("week12_transactions.csv")
print("Shape:", df.shape)
print("Total RWF:", df["amount_rwf"].sum())

Shape: (2000, 7)
Total RWF: 119876000


In [5]:
import os
os.chdir("/content/Big_Data_Week_12_Notebook")
print("Now in:", os.getcwd())

Now in: /content/Big_Data_Week_12_Notebook


In [6]:
import os
print(os.getcwd())
print(os.listdir('.'))

/content/Big_Data_Week_12_Notebook
['Lab 6.pdf', '.git', 'week12_transactions.xlsx', 'week12_api_sample.json', 'week12_transactions.csv', 'README.md']


In [8]:
import pandas as pd

df = pd.read_csv("week12_transactions.csv")
print("Shape:", df.shape)                    # expect (2000, 7)
print("Total RWF:", df["amount_rwf"].sum())  # expect 119876000
print("Districts:", df["district"].nunique())
print("Flagged:", df["flagged"].sum())        # expect 11

Shape: (2000, 7)
Total RWF: 119876000
Districts: 8
Flagged: 11


In [9]:
import os
print("cwd:", os.getcwd())

import pandas as pd
df = pd.read_csv("week12_transactions.csv")
print("Shape:", df.shape)
print("Total RWF:", df["amount_rwf"].sum())
print("Districts:", df["district"].nunique())
print("Flagged:", df["flagged"].sum())

cwd: /content/Big_Data_Week_12_Notebook
Shape: (2000, 7)
Total RWF: 119876000
Districts: 8
Flagged: 11


In [10]:
import sqlite3

# Build DB
conn = sqlite3.connect("momo.db")
df.to_sql("transactions", conn, if_exists="replace", index=False)

# 1.1a — top 5 sends > 100k
print("=== 1.1a: Top 5 sends > 100k ===")
print(pd.read_sql("""
SELECT txn_id, amount_rwf, flagged
FROM transactions
WHERE txn_type = 'send' AND amount_rwf > 100000
ORDER BY amount_rwf DESC LIMIT 5
""", conn).to_string(index=False))

# 1.1b — flagged across all sends > 100k
print("\n=== 1.1b: Flagged across all sends > 100k ===")
print(pd.read_sql("""
SELECT COUNT(*) AS n_sends, SUM(flagged) AS flagged
FROM transactions
WHERE txn_type = 'send' AND amount_rwf > 100000
""", conn).to_string(index=False))

# 1.2a — per txn_type
print("\n=== 1.2a: Per txn_type ===")
print(pd.read_sql("""
SELECT txn_type, COUNT(*) AS n_txns,
       SUM(amount_rwf) AS total_rwf,
       ROUND(AVG(amount_rwf), 2) AS avg_rwf
FROM transactions GROUP BY txn_type ORDER BY total_rwf DESC
""", conn).to_string(index=False))

# 1.2b — per district
print("\n=== 1.2b: Per district ===")
print(pd.read_sql("""
SELECT district, SUM(amount_rwf) AS total_rwf
FROM transactions GROUP BY district ORDER BY total_rwf DESC
""", conn).to_string(index=False))

# 1.3 — SQL vs Pandas hours
sql_hour = pd.read_sql(
    "SELECT hour, COUNT(*) AS n FROM transactions GROUP BY hour ORDER BY hour", conn)
pandas_hour = df.groupby("hour").size().reset_index(name="n").sort_values("hour")
merged = sql_hour.merge(pandas_hour, on="hour", suffixes=("_sql", "_pandas"))
merged["match"] = merged["n_sql"] == merged["n_pandas"]

print("\n=== 1.3: SQL vs Pandas per hour ===")
print("All hours match:", merged["match"].all())
print("Busiest hour:", int(merged.loc[merged["n_sql"].idxmax(), "hour"]))
print("Max txns:", int(merged["n_sql"].max()))

=== 1.1a: Top 5 sends > 100k ===
txn_id  amount_rwf  flagged
T01708      140000        0
T00364      139000        0
T00895      139000        0
T00899      139000        0
T01314      139000        0

=== 1.1b: Flagged across all sends > 100k ===
 n_sends  flagged
     255       11

=== 1.2a: Per txn_type ===
txn_type  n_txns  total_rwf  avg_rwf
    send     868   62049000 71485.02
 paybill     330   29970000 90818.18
 receive     489   27024000 55263.80
 airtime     313     833000  2661.34

=== 1.2b: Per district ===
  district  total_rwf
   Muhanga   21076000
    Gasabo   20738000
      Huye   18736000
Nyarugenge   18091000
  Kicukiro   11338000
   Musanze   10490500
    Rubavu   10110500
 Nyagatare    9296000

=== 1.3: SQL vs Pandas per hour ===
All hours match: True
Busiest hour: 20
Max txns: 185


In [11]:
import json

json_string = '''
{
  "city": "Kigali",
  "readings": [22.1, 24.5, 25.0, 26.3, 23.8, 21.9]
}
'''
parsed = json.loads(json_string)
print(f"City: {parsed['city']}")
print(f"Number of readings: {len(parsed['readings'])}")
print(f"Average temperature: {sum(parsed['readings'])/len(parsed['readings']):.2f} °C")

City: Kigali
Number of readings: 6
Average temperature: 23.93 °C


In [12]:
import json, pandas as pd

with open("week12_api_sample.json") as f:
    data = json.load(f)

times = data["hourly"]["time"]
temps = data["hourly"]["temperature_2m"]

print("Hours:", len(times))
print("Max:", max(temps), "°C")
print("Min:", min(temps), "°C")
print("Hottest at:", times[temps.index(max(temps))])

df_w = pd.DataFrame({"time": times, "temp_c": temps})
print(df_w.describe())
print("Hours ≥ 25 °C:", int((df_w["temp_c"] >= 25).sum()))

Hours: 24
Max: 27.9 °C
Min: 15.9 °C
Hottest at: 2026-11-22T14:00
          temp_c
count  24.000000
mean   21.137500
std     4.104696
min    15.900000
25%    17.650000
50%    19.900000
75%    25.025000
max    27.900000
Hours ≥ 25 °C: 7
